## 🏆 FIFA World Cup 2026 Data Pipeline - Hybrid Approach

**Data Sources:**
* ⭐ **SportScore API** (https://sportscore.com) - Players, Teams, Group Standings
* 🌐 **worldcup26.ir API** - Matches (104 complete), Stadiums (16 venues)

**📋 Execution Order (Run cells in sequence):**
1. ⚙️ **Imports and Config** - Load API key from secrets (`fifa_pipeline` scope)
2. 1️⃣ **Fetch Players** - 80 top performers from SportScore
3. 2️⃣ **Fetch Matches** - 104 complete matches from worldcup26.ir
4. 3️⃣ **Fetch Group Standings** - 60 standings from SportScore
5. 4️⃣ **Fetch Stadiums** - 16 venues from worldcup26.ir
6. 5️⃣ **Save All Tables** - Write to Unity Catalog
7. 📊 **Final Verification** - Check all tables

**Target:** `singhal.fifa_worldcup_bronze.*`

---

In [0]:
# FIFA World Cup 2026 Data Pipeline - SportScore API
import requests
import json
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import col

# Configuration
CATALOG = "singhal"
SCHEMA = "fifa_worldcup_bronze"
SPORTSCORE_API_HOST = "https://sportscore.com"

# Get SportScore API Key from Databricks Secrets (secure!)
SPORTSCORE_API_KEY = dbutils.secrets.get(scope="fifa_pipeline", key="api_football_key")
SPORTSCORE_HEADERS = {"X-API-Key": SPORTSCORE_API_KEY}

ingested_at = datetime.now(timezone.utc)

print("✅ Configuration loaded")
print(f"   Catalog: {CATALOG}")
print(f"   Schema: {SCHEMA}")
print(f"   API: SportScore (free tier, 10K requests/day)")

In [0]:
# ✅ FETCH PLAYERS FROM SPORTSCORE API (Single Source for Player Stats)
# This is the ONLY source for player data - no mixing with worldcup26.ir
# worldcup26.ir is ONLY used for matches, stadiums, and fixtures

print(f"🔍 1️⃣ Fetching FIFA World Cup 2026 Players from SportScore...")
print(f"="*80)
print(f"   Source: SportScore API (top 50 scorers + assisters)")
print(f"   Note: worldcup26.ir NOT used for player stats (prevents double-counting)")

try:
    # Fetch top scorers (goals)
    goals_response = requests.get(
        f"{SPORTSCORE_API_HOST}/api/widget/topscorers/",
        params={"sport": "football", "slug": "fifa-world-cup", "stat": "goals", "limit": 50},
        headers=SPORTSCORE_HEADERS,
        timeout=30
    )
    
    # Fetch top assisters
    assists_response = requests.get(
        f"{SPORTSCORE_API_HOST}/api/widget/topscorers/",
        params={"sport": "football", "slug": "fifa-world-cup", "stat": "assists", "limit": 50},
        headers=SPORTSCORE_HEADERS,
        timeout=30
    )
    
    if goals_response.status_code == 200 and assists_response.status_code == 200:
        goals_data = goals_response.json()
        assists_data = assists_response.json()
        
        scorers = goals_data.get('scorers', [])
        assisters = assists_data.get('scorers', [])
        
        # Merge player data by player_slug (unique ID)
        players_dict = {}
        
        # Add scorers
        for player in scorers:
            player_id = player['player_slug']
            players_dict[player_id] = {
                'player_id': player_id,
                'player_name': player['player'],
                'player_logo': player.get('player_logo'),  # ✅ Player image URL
                'team_name': player['team'],  # Team name from API
                'goals_scored': player.get('goals', 0),
                'assists': player.get('assists', 0),
                'matches_played': player.get('matches', 0),
                'minutes_played': player.get('minutes', 0),
                'rating': player.get('rating', 0),
            }
        
        # Update with assist data
        for player in assisters:
            player_id = player['player_slug']
            if player_id in players_dict:
                players_dict[player_id]['assists'] = max(
                    players_dict[player_id]['assists'],
                    player.get('assists', 0)
                )
            else:
                players_dict[player_id] = {
                    'player_id': player_id,
                    'player_name': player['player'],
                    'player_logo': player.get('player_logo'),  # ✅ Player image URL
                    'team_name': player['team'],
                    'goals_scored': player.get('goals', 0),
                    'assists': player.get('assists', 0),
                    'matches_played': player.get('matches', 0),
                    'minutes_played': player.get('minutes', 0),
                    'rating': player.get('rating', 0),
                }
        
        players_raw = list(players_dict.values())
        print(f"✅ Fetched {len(players_raw)} unique players")
        print(f"   Top scorer: {players_raw[0]['player_name']} ({players_raw[0]['goals_scored']} goals)")
        
    else:
        print(f"❌ Error fetching players")
        players_raw = []
        
except Exception as e:
    print(f"❌ Exception: {type(e).__name__}: {str(e)}")
    players_raw = []

In [0]:
# Fetch FIFA World Cup 2026 matches from worldcup26.ir (complete 104 matches)
print(f"\n🔍 2⃣ Fetching FIFA World Cup 2026 Matches from worldcup26.ir...")
print(f"="*80)

WORLDCUP_API_HOST = "https://worldcup26.ir"

try:
    # Fetch all 104 matches from worldcup26.ir
    matches_response = requests.get(
        f"{WORLDCUP_API_HOST}/get/games",
        timeout=30
    )
    matches_response.raise_for_status()
    
    # ✅ FIX: Explicitly set UTF-8 encoding to prevent corruption of special characters
    # Without this, requests auto-detects wrong encoding (ISO-8859-1) causing:
    # "Julián" → "Jvlian", "Cody Gakpo" → "Kvdi Khakpv", etc.
    matches_response.encoding = 'utf-8'
    matches_data = matches_response.json()
    
    # The response has a 'games' key with list of match objects
    worldcup_matches = matches_data.get("games", [])
    
    # Helper function to safely convert to int
    def safe_int(value, default=0):
        if value is None or value == "null" or value == "":
            return default
        try:
            return int(value)
        except (ValueError, TypeError):
            return default
    
    # Transform matches
    matches_raw = []
    for match in worldcup_matches:
        matches_raw.append({
            'match_id': str(match.get("id")),
            'home_team_id': str(match.get("home_team_id")) if match.get("home_team_id") not in [None, "null", "", "0"] else None,
            'away_team_id': str(match.get("away_team_id")) if match.get("away_team_id") not in [None, "null", "", "0"] else None,
            'home_team_name': match.get("home_team_name_en"),
            'away_team_name': match.get("away_team_name_en"),
            'home_score': safe_int(match.get("home_score")),
            'away_score': safe_int(match.get("away_score")),
            'home_scorers': match.get("home_scorers") if match.get("home_scorers") != "null" else None,
            'away_scorers': match.get("away_scorers") if match.get("away_scorers") != "null" else None,
            'home_penalty_score': safe_int(match.get("home_penalty_score")),
            'away_penalty_score': safe_int(match.get("away_penalty_score")),
            'home_penalty_scorers': match.get("home_penalty_scorers") if match.get("home_penalty_scorers") not in [None, "null", ""] else None,
            'away_penalty_scorers': match.get("away_penalty_scorers") if match.get("away_penalty_scorers") not in [None, "null", ""] else None,
            'home_penalty_misses': match.get("home_penalty_misses") if match.get("home_penalty_misses") not in [None, "null", ""] else None,
            'away_penalty_misses': match.get("away_penalty_misses") if match.get("away_penalty_misses") not in [None, "null", ""] else None,
            'group': match.get("group"),
            'matchday': match.get("matchday"),
            'match_type': match.get("type"),
            'local_date': match.get("local_date"),
            'stadium_id': str(match.get("stadium_id")),
            'finished': match.get("finished"),
            'time_elapsed': match.get("time_elapsed"),
            'home_team_label': match.get("home_team_label"),
            'away_team_label': match.get("away_team_label"),
        })
    
    print(f"✅ Fetched {len(matches_raw)} FIFA World Cup matches")
    if matches_raw:
        print(f"   Sample: {matches_raw[0]['home_team_name']} vs {matches_raw[0]['away_team_name']} ({matches_raw[0]['home_score']}-{matches_raw[0]['away_score']})")
        
except Exception as e:
    print(f"❌ Exception: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()
    matches_raw = []

In [0]:
# Fetch stadiums from worldcup26.ir (16 venues)
print(f"\n🔍 4⃣ Fetching Stadiums from worldcup26.ir...")
print(f"="*80)

WORLDCUP_API_HOST = "https://worldcup26.ir"

try:
    stadiums_response = requests.get(
        f"{WORLDCUP_API_HOST}/get/stadiums",
        timeout=30
    )
    stadiums_response.raise_for_status()
    stadiums_data = stadiums_response.json()
    
    # The response has a 'stadiums' key with list of stadium objects
    stadiums_raw = stadiums_data.get("stadiums", [])
    
    print(f"✅ Fetched {len(stadiums_raw)} stadiums")
    if stadiums_raw:
        print(f"   Sample: {stadiums_raw[0].get('name_en')} (capacity: {stadiums_raw[0].get('capacity')})")
    
    stadiums_action = "fetched"
        
except Exception as e:
    print(f"❌ Exception: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()
    stadiums_raw = []
    stadiums_action = "skip"

In [0]:
# Fetch FIFA World Cup 2026 group standings
print(f"\n🔍 3⃣ Fetching FIFA World Cup 2026 Group Standings...")
print(f"="*80)

try:
    standings_response = requests.get(
        f"{SPORTSCORE_API_HOST}/api/widget/standings/",
        params={"sport": "football", "slug": "fifa-world-cup"},
        headers=SPORTSCORE_HEADERS,
        timeout=30
    )
    
    if standings_response.status_code == 200:
        standings_data = standings_response.json()
        
        # SportScore returns standings grouped by tables (groups)
        standings_sections = standings_data.get('tables', [])
        
        standings_raw = []
        teams_raw = []
        team_id_counter = 1
        team_name_to_id = {}  # Map team names to IDs
        
        for section in standings_sections:
            group_name = section.get('group', 'Unknown')  # e.g., "Group 1"
            teams = section.get('rows', [])
            
            for team in teams:
                team_name = team.get('team', '')
                
                # Create team entry (if not exists)
                if team_name not in team_name_to_id:
                    team_id = f"team_{team_id_counter}"
                    team_id_counter += 1
                    team_name_to_id[team_name] = team_id
                    
                    teams_raw.append({
                        'team_id': team_id,
                        'team_name': team_name,
                        'group_name': group_name,
                        'team_logo': team.get('team_logo', ''),
                    })
                else:
                    team_id = team_name_to_id[team_name]
                
                # Add standings entry (use correct field names from API)
                standings_raw.append({
                    'team_id': team_id,
                    'team_name': team_name,
                    'group_name': group_name,
                    'matches_played': team.get('p', 0),  # 'p' = matches played
                    'wins': team.get('w', 0),  # 'w' = wins
                    'draws': team.get('d', 0),  # 'd' = draws
                    'losses': team.get('l', 0),  # 'l' = losses
                    'goals_for': team.get('gf', 0),  # 'gf' = goals for
                    'goals_against': team.get('ga', 0),  # 'ga' = goals against
                    'goal_difference': team.get('gd', 0),  # 'gd' = goal difference
                    'points': team.get('pts', 0),  # 'pts' = points
                })
        
        print(f"✅ Fetched {len(standings_raw)} group standings entries")
        print(f"✅ Extracted {len(teams_raw)} unique teams")
        if standings_raw:
            print(f"   Top team: {standings_raw[0]['team_name']} ({standings_raw[0]['points']} pts)")
        
    else:
        print(f"❌ Error fetching standings: {standings_response.status_code}")
        standings_raw = []
        teams_raw = []
        
except Exception as e:
    print(f"❌ Exception: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()
    standings_raw = []
    teams_raw = []

In [0]:
# Save all fetched data to Unity Catalog tables
print(f"\n🔍 5⃣ Saving All Tables to Unity Catalog...")
print(f"="*80)

# 🛡️ SAFETY CHECK: Validate data quality before overwriting
MIN_PLAYERS = 50      # Expect at least 50 players (top scorers/assisters)
MIN_TEAMS = 40        # Expect at least 40 teams
MIN_MATCHES = 90      # Expect at least 90 matches (104 total)
MIN_STANDINGS = 50    # Expect at least 50 standings entries (60 total)
MIN_STADIUMS = 15     # Expect at least 15 stadiums (16 total)

data_quality_ok = True
errors = []

if len(players_raw) < MIN_PLAYERS:
    errors.append(f"⚠️ Players: {len(players_raw)} < {MIN_PLAYERS} minimum")
    data_quality_ok = False

if len(teams_raw) < MIN_TEAMS:
    errors.append(f"⚠️ Teams: {len(teams_raw)} < {MIN_TEAMS} minimum")
    data_quality_ok = False

if len(matches_raw) < MIN_MATCHES:
    errors.append(f"⚠️ Matches: {len(matches_raw)} < {MIN_MATCHES} minimum")
    data_quality_ok = False

if len(standings_raw) < MIN_STANDINGS:
    errors.append(f"⚠️ Standings: {len(standings_raw)} < {MIN_STANDINGS} minimum")
    data_quality_ok = False

if len(stadiums_raw) < MIN_STADIUMS:
    errors.append(f"⚠️ Stadiums: {len(stadiums_raw)} < {MIN_STADIUMS} minimum")
    data_quality_ok = False

if not data_quality_ok:
    print(f"\n🚨 DATA QUALITY CHECK FAILED!")
    print(f"="*80)
    for error in errors:
        print(f"   {error}")
    print(f"\n❌ SKIPPING TABLE SAVE to prevent data loss!")
    print(f"   Existing tables preserved. Fix API issues and re-run.")
    print(f"="*80)
    raise Exception("Data quality check failed - insufficient records fetched")

print(f"✅ Data quality check passed:")
print(f"   Players: {len(players_raw)} (min: {MIN_PLAYERS})")
print(f"   Teams: {len(teams_raw)} (min: {MIN_TEAMS})")
print(f"   Matches: {len(matches_raw)} (min: {MIN_MATCHES})")
print(f"   Standings: {len(standings_raw)} (min: {MIN_STANDINGS})")
print(f"   Stadiums: {len(stadiums_raw)} (min: {MIN_STADIUMS})")
print(f"\n💾 Proceeding with table save...")

tables_saved = []

# 1. Save Players
if players_raw:
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
    
    # Define explicit schema (use LongType to match existing BIGINT columns)
    player_schema = StructType([
        StructField("player_id", StringType(), True),
        StructField("player_name", StringType(), True),
        StructField("player_logo", StringType(), True),  # ✅ Player image URL
        StructField("team_id", StringType(), True),
        StructField("team_name", StringType(), True),
        StructField("goals_scored", LongType(), True),
        StructField("assists", LongType(), True),
        StructField("matches_played", LongType(), True),
        StructField("minutes_played", LongType(), True),
        StructField("rating", LongType(), True),
        StructField("ingested_at", TimestampType(), True),
    ])
    
    player_rows = []
    for player in players_raw:
        # Join with teams to get team_id
        team_id = None
        for team in teams_raw:
            if team['team_name'] == player['team_name']:
                team_id = team['team_id']
                break
        
        player_rows.append((
            str(player['player_id']),
            str(player['player_name']),
            str(player.get('player_logo')) if player.get('player_logo') else None,  # ✅ Player image
            str(team_id) if team_id else None,
            str(player['team_name']),
            int(player['goals_scored']),
            int(player['assists']),
            int(player['matches_played']),
            int(player['minutes_played']),
            int(player['rating']),
            ingested_at,
        ))
    
    spark.createDataFrame(player_rows, schema=player_schema).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.players")
    print(f"✅ Saved {len(player_rows)} players to {CATALOG}.{SCHEMA}.players")
    tables_saved.append('players')
else:
    print(f"⚠️ No players data to save")

# 2. Save Teams
if teams_raw:
    team_rows = []
    for team in teams_raw:
        team_rows.append(Row(
            team_id=team['team_id'],
            team_name=team['team_name'],
            group_name=team['group_name'],
            team_logo=team.get('team_logo', ''),
            ingested_at=ingested_at,
        ))
    
    spark.createDataFrame(team_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.teams")
    print(f"✅ Saved {len(team_rows)} teams to {CATALOG}.{SCHEMA}.teams")
    tables_saved.append('teams')
else:
    print(f"⚠️ No teams data to save")

# 3. Save Matches
if matches_raw:
    match_rows = []
    for match in matches_raw:
        match_rows.append(Row(
            match_id=match['match_id'],
            home_team_id=match['home_team_id'],
            away_team_id=match['away_team_id'],
            home_team_name=match['home_team_name'],
            away_team_name=match['away_team_name'],
            home_score=match['home_score'],
            away_score=match['away_score'],
            home_scorers=match['home_scorers'],
            away_scorers=match['away_scorers'],
            home_penalty_score=match['home_penalty_score'],
            away_penalty_score=match['away_penalty_score'],
            home_penalty_scorers=match['home_penalty_scorers'],
            away_penalty_scorers=match['away_penalty_scorers'],
            home_penalty_misses=match['home_penalty_misses'],
            away_penalty_misses=match['away_penalty_misses'],
            group=match['group'],
            matchday=match['matchday'],
            match_type=match['match_type'],
            local_date=match['local_date'],
            stadium_id=match['stadium_id'],
            finished=match['finished'],
            time_elapsed=match['time_elapsed'],
            home_team_label=match['home_team_label'],
            away_team_label=match['away_team_label'],
            ingested_at=ingested_at,
        ))
    
    spark.createDataFrame(match_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.matches")
    print(f"✅ Saved {len(match_rows)} matches to {CATALOG}.{SCHEMA}.matches")
    tables_saved.append('matches')
else:
    print(f"⚠️ No matches data to save")

# 4. Save Group Standings
if standings_raw:
    standing_rows = []
    for standing in standings_raw:
        standing_rows.append(Row(
            team_id=standing['team_id'],
            team_name=standing['team_name'],
            group_name=standing['group_name'],
            matches_played=standing['matches_played'],
            wins=standing['wins'],
            draws=standing['draws'],
            losses=standing['losses'],
            goals_for=standing['goals_for'],
            goals_against=standing['goals_against'],
            goal_difference=standing['goal_difference'],
            points=standing['points'],
            ingested_at=ingested_at,
        ))
    
    spark.createDataFrame(standing_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.group_standings")
    print(f"✅ Saved {len(standing_rows)} standings to {CATALOG}.{SCHEMA}.group_standings")
    tables_saved.append('group_standings')
else:
    print(f"⚠️ No standings data to save")

print(f"\n{'='*80}")
print(f"✅ PIPELINE COMPLETE!")
print(f"{'='*80}")
print(f"\n📊 Tables saved: {', '.join(tables_saved)}")
print(f"\n📄 Summary:")
for table in tables_saved:
    count_result = spark.sql(f"SELECT COUNT(*) as count FROM {CATALOG}.{SCHEMA}.{table}").first()
    count = count_result[0]  # Get the first column value
    print(f"   ✔️ {table}: {count} rows")

# 5. Save Stadiums (if fetched)
if stadiums_action == "fetched" and stadiums_raw:
    stadium_rows = []
    for stadium in stadiums_raw:
        stadium_rows.append(Row(
            stadium_id=str(stadium.get("id")),
            name_en=stadium.get("name_en"),
            name_fa=stadium.get("name_fa"),
            fifa_name=stadium.get("fifa_name"),
            city_en=stadium.get("city_en"),
            country_en=stadium.get("country_en"),
            capacity=int(stadium.get("capacity", 0)),
            ingested_at=ingested_at,
        ))
    
    spark.createDataFrame(stadium_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.stadiums")
    print(f"✅ Saved {len(stadium_rows)} stadiums to {CATALOG}.{SCHEMA}.stadiums")
    tables_saved.append('stadiums')
else:
    try:
        stadium_count = spark.sql(f"SELECT COUNT(*) as count FROM {CATALOG}.{SCHEMA}.stadiums").first()[0]
        print(f"   ℹ️  stadiums: {stadium_count} rows (kept from previous run)")
    except:
        print(f"   ⚠️  stadiums: Not fetched or table not found")

## 🛡️ Data Safety Guardrails

### ⚠️ **Problem This Solves**
If API calls fail (network issues, rate limits, key expiration), the notebook would fetch 0 records and **overwrite existing good data with empty tables** — catastrophic data loss!

---

### ✅ **Safety Checks Implemented**

#### 1️⃣ **Pre-Save Validation (Cell 7)**
```python
MIN_PLAYERS = 50      # Expect at least 50 players
MIN_TEAMS = 40        # Expect at least 40 teams  
MIN_MATCHES = 90      # Expect at least 90 matches
MIN_STANDINGS = 50    # Expect at least 50 standings
MIN_STADIUMS = 15     # Expect at least 15 stadiums
```

**Behavior:**
* ✅ If **ALL 5 tables** meet minimums → Proceed with overwrite
* ❌ If **ANY table** below minimum → **ABORT** and preserve existing tables
* 🛡️ Protects: players, teams, matches, standings, stadiums
* 📊 Prevents empty API responses from destroying data

#### 2️⃣ **Backfill Validation (Cell 8)**
* Checks matches table has 90+ records before backfilling
* Validates 100+ unique scorers found
* Skips backfill if data looks incomplete

---

### 📋 **What Happens When API Fails?**

**Before (Dangerous):**
```
API fails → players_raw = [] → Overwrite table → 💥 Data loss!
```

**After (Safe):**
```
API fails → players_raw = [] → Quality check fails → ❌ Aborts
                            ↓
                    Existing data preserved ✅
```

---

### 🔄 **Recovery Steps If Notebook Aborts**

1. **Check API Status**
   - Verify API endpoints are accessible
   - Check API key in secrets: `dbutils.secrets.get("fifa_pipeline", "api_football_key")`
   - Test API calls manually in temp notebook

2. **Review Error Messages**
   - Notebook will show which validation failed
   - Check actual counts vs minimums

3. **Adjust Thresholds (if needed)**
   - If World Cup ends and fewer matches remain, lower `MIN_MATCHES`
   - Thresholds are in Cell 7, easy to modify

4. **Re-run After Fix**
   - Fix API issue
   - Re-run notebook
   - Data only overwrites if quality checks pass

---

### 🚀 **Future Enhancement: Change Data Capture (CDC)**

For production pipelines, consider:
* **Incremental loads** instead of full overwrites
* **MERGE operations** to update only changed records
* **Staging tables** with swap-on-success pattern
* **Audit logging** with timestamps and row counts
* **Alerting** on data quality failures

In [0]:
# Bronze Layer Summary - Raw Data Only (No Transformations)
print(f"🏆 FIFA WORLD CUP 2026 - BRONZE LAYER (Raw API Data)")
print(f"="*80)
print(f"   Note: goal_events is derived in SILVER layer via dbt (from matches table)")
print(f"="*80)

# 1. PLAYERS (80 top performers)
print(f"\n👥 PLAYERS (80 top performers)")
print(f"-"*80)
players_df = spark.sql(f"""
    SELECT player_id, player_name, team_name, goals_scored, assists, matches_played
    FROM {CATALOG}.{SCHEMA}.players
    ORDER BY goals_scored DESC, assists DESC
    LIMIT 10
""")
players_df.show(truncate=False)

# 2. TEAMS (48 teams from standings)
print(f"\n📏 TEAMS (48 teams)")
print(f"-"*80)
teams_df = spark.sql(f"""
    SELECT team_id, team_name, group_name
    FROM {CATALOG}.{SCHEMA}.teams
    ORDER BY group_name, team_name
    LIMIT 10
""")
teams_df.show(truncate=False)

# 3. MATCHES (104 complete matches from worldcup26.ir)
print(f"\n⚽ MATCHES (104 complete matches)")
print(f"-"*80)
matches_df = spark.sql(f"""
    SELECT match_id, home_team_name, away_team_name, home_score, away_score, 
           match_type, local_date, finished
    FROM {CATALOG}.{SCHEMA}.matches
    ORDER BY local_date DESC
    LIMIT 10
""")
matches_df.show(truncate=False)

# 4. GROUP STANDINGS (60 entries across 12 groups)
print(f"\n📊 GROUP STANDINGS (60 entries)")
print(f"-"*80)
standings_df = spark.sql(f"""
    SELECT group_name, team_name, points, wins, draws, losses, 
           goals_for, goals_against, goal_difference
    FROM {CATALOG}.{SCHEMA}.group_standings
    ORDER BY group_name, points DESC, goal_difference DESC
    LIMIT 12
""")
standings_df.show(truncate=False)

# 5. STADIUMS (16 venues from worldcup26.ir)
print(f"\n🏟️ STADIUMS (16 venues)")
print(f"-"*80)
stadiums_df = spark.sql(f"""
    SELECT name_en as name, city_en as city, country_en as country, capacity
    FROM {CATALOG}.{SCHEMA}.stadiums
    ORDER BY capacity DESC
    LIMIT 5
""")
stadiums_df.show(truncate=False)

print(f"\n{'='*80}")
print(f"✅ BRONZE LAYER COMPLETE - 5 RAW TABLES")
print(f"{'='*80}")
print(f"\n📦 Data Sources:")
print(f"   ⭐ SportScore API: players, teams, group_standings")
print(f"   🌐 worldcup26.ir API: matches, stadiums")
print(f"\n🔄 Next Step: dbt Silver Layer")
print(f"   silver.goal_events will be derived from bronze.matches")